### Preliminary analysis of the GRI

Using standard models for survival analysis with competing risks such Cause specific Cox PH and Fine and Grey. 

We will run it over two datasets, one would be the subset of qpis (quality measures for colorectal cancer), 
and secondly over the whole clinical data, where there are potential predictors. The dataset has been preprocessed in "prpeprocess-1.ipynb", where expert knowledge from Joanne Edwards' lab was considered to substract the relevant columns and rows, plus any further decisions taken. 

Hopefully the second dataset would allow give us hints of future qpis. 

Regarding missing values, they need to be imputed... the continuous values standarized, etc. 

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt

# sksurv
from sksurv.util import Surv
from sksurv.linear_model import CoxPHSurvivalAnalysis
# scikit-learn
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import make_scorer

# Iterative inputer (for mice)
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import IterativeImputer
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_columns", None)



### 5-fold cross-validation


We can use MICE for inputation

In [4]:
## prepare the data

df = pd.read_csv("data/GRI_preprocess-1.csv")

# define time and status
time, status = "DFS_months_2020", "Cmprsk_Status"
# create surv object 
#y = Surv.from_dataframe(time, status, df) # does not work because it expects binary
X = df.drop(columns=[time, status])
T_all = df[time].astype(float).to_numpy()
D_all = df[status].astype(int).to_numpy()



NameError: name 'pd' is not defined

We can use the pipe to introduce preprocessing with mice and scaling. 



In [3]:
def modelling_pipe(model):
    # select the numeric
    num_sel = make_column_selector(dtype_include=np.number)
    # select the categorical
    cat_sel = make_column_selector(dtype_include=["object", "category", "string", "bool"])

    num = Pipeline([
        # Using the equivalent to Mice in R
        ("mice", IterativeImputer(random_state=42, sample_posterior=True, max_iter=10, initial_strategy="median")),
        ("scaler", StandardScaler()),
        ])
    cat = Pipeline([
        # since it is categorical we impute with the most frequent
        ("mode", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore", sparse=False))
    ])
    prep = ColumnTransformer([
        ("num", num, num_sel),
        ("cat", cat, cat_sel)], 
        remainder="drop")
    
    return Pipeline([("prep", prep),("model", model)])
    

In [2]:
any_event = (D_all != 0).astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
splits = list(cv.split(np.arange(len(df)), y=any_event))

NameError: name 'D_all' is not defined